# 08 ETCCDI Spatial and Trend Analysis

This notebook continues from `07_fast_etccdi_index_bias_correction` and creates reviewer-ready outputs: spatial change maps, model agreement diagnostics, Mann-Kendall trend tests, Sen slopes, FDR-adjusted significance, tables, figures, logs, and a phase summary.


## Cell 1 - Setup and Paths


In [ ]:
from pathlib import Path
import math
import re
import warnings

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib as mpl

warnings.filterwarnings('ignore')

ROOT = (Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve())
ETCCDI_ROOT = ROOT / 'output' / 'etccdi_fast'
CORR_DIR = ETCCDI_ROOT / 'corrected_indices'
ZONE_TABLE = ETCCDI_ROOT / 'tables' / 'fast_etccdi_zone_annual_means.csv'
ZONES_FILE = ROOT / 'output' / 'zones' / 'hydroclimatic_zones_SA.nc'

OUT_ROOT = ROOT / 'output' / 'etccdi_spatial_trends'
TABLE_DIR = OUT_ROOT / 'tables'
FIG_DIR = OUT_ROOT / 'figures'
MAP_DIR = OUT_ROOT / 'maps'
LOG_DIR = OUT_ROOT / 'logs'
for d in [TABLE_DIR, FIG_DIR, MAP_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

FULL_MODELS = ['CanESM5', 'GFDL-ESM4', 'INM-CM5-0', 'IPSL-CM6A-LR', 'MPI-ESM1-2-HR']
PRECIP_ONLY_MODELS = ['CESM2']
MODELS = FULL_MODELS + PRECIP_ONLY_MODELS
SCENARIOS = ['historical', 'ssp245', 'ssp585']
FUTURE_SCENARIOS = ['ssp245', 'ssp585']
BASELINE = (1985, 2014)
PERIODS = {'near_future_2021_2060': (2021, 2060), 'far_future_2061_2100': (2061, 2100)}

KEY_INDICES = ['PRCPTOT', 'RX1day', 'CDD', 'TXx', 'TNn']
PRECIP_INDICES = ['PRCPTOT', 'RX1day', 'RX5day', 'SDII', 'R10mm', 'R20mm', 'CDD', 'CWD']
TEMP_INDICES = ['TXx', 'TXn', 'TNx', 'TNn', 'DTR', 'SU', 'TR', 'FD', 'ID']
INDEX_UNITS = {'PRCPTOT': 'mm', 'RX1day': 'mm', 'RX5day': 'mm', 'SDII': 'mm day-1', 'R10mm': 'days', 'R20mm': 'days', 'CDD': 'days', 'CWD': 'days', 'TXx': 'degC', 'TXn': 'degC', 'TNx': 'degC', 'TNn': 'degC', 'DTR': 'degC', 'SU': 'days', 'TR': 'days', 'FD': 'days', 'ID': 'days'}
INDEX_TITLES = {'PRCPTOT': 'Annual wet-day precipitation', 'RX1day': 'Maximum 1-day precipitation', 'CDD': 'Consecutive dry days', 'TXx': 'Annual maximum of daily Tmax', 'TNn': 'Annual minimum of daily Tmin'}
SCENARIO_LABELS = {'historical': 'Historical', 'ssp245': 'SSP2-4.5', 'ssp585': 'SSP5-8.5'}
ZONE_IDS = list(range(7))
ZONE_LABELS = {z: f'Z{z+1}' for z in ZONE_IDS}
print('Input:', CORR_DIR)
print('Output:', OUT_ROOT)


## Cell 2 - Helper Functions


In [ ]:
def standardise_xy(ds):
    rename = {}
    for cand in ['latitude', 'y']:
        if cand in ds.coords or cand in ds.dims:
            rename[cand] = 'lat'
    for cand in ['longitude', 'x']:
        if cand in ds.coords or cand in ds.dims:
            rename[cand] = 'lon'
    if rename:
        ds = ds.rename(rename)
    if 'lon' in ds.coords and float(ds.lon.max()) > 180:
        ds = ds.assign_coords(lon=((ds.lon + 180) % 360) - 180).sortby('lon')
    if 'lat' in ds.coords:
        ds = ds.sortby('lat')
    if 'lon' in ds.coords:
        ds = ds.sortby('lon')
    return ds

def first_data_var(ds):
    return list(ds.data_vars)[0] if ds.data_vars else None

def force_numeric_da(da):
    da = standardise_xy(da.squeeze(drop=True))
    if np.issubdtype(da.dtype, np.timedelta64):
        da = da / np.timedelta64(1, 'D')
    elif not np.issubdtype(da.dtype, np.number):
        da = da.astype('float64')
    keep = {c: da.coords[c] for c in ['lat', 'lon', 'year'] if c in da.coords}
    da = da.reset_coords(drop=True)
    if keep:
        da = da.assign_coords(keep)
    return da.astype('float32')

def corrected_index_path(model, scenario, index_name):
    return CORR_DIR / scenario / model / f'{model}_{scenario}_{index_name}_annual_corrected.nc'

def load_corrected_index(model, scenario, index_name):
    path = corrected_index_path(model, scenario, index_name)
    if not path.exists():
        return None
    try:
        with xr.open_dataset(path) as ds:
            var = index_name if index_name in ds.data_vars else first_data_var(ds)
            return None if var is None else force_numeric_da(ds[var]).load()
    except Exception as exc:
        failures.append(f'LOAD_FAIL|{model}|{scenario}|{index_name}|{type(exc).__name__}: {exc}')
        return None

def period_mean(da, start, end):
    if da is None or 'year' not in da.coords:
        return None
    sub = da.sel(year=slice(start, end))
    return None if sub.sizes.get('year', 0) == 0 else sub.mean('year', skipna=True)

def safe_percent_change(change, baseline):
    return 100.0 * change / baseline.where(np.abs(baseline) > 1e-6)

def to_zone_grid(da, zone_da):
    return da.interp(lat=zone_da.lat, lon=zone_da.lon, method='nearest')


## Cell 3 - Validate Inputs


In [ ]:
failures = []
for p in [CORR_DIR, ZONE_TABLE, ZONES_FILE]:
    if not p.exists():
        raise FileNotFoundError(p)

zone_summary = pd.read_csv(ZONE_TABLE)
required = {'scenario', 'model', 'index', 'year', 'zone', 'zone_id', 'zone_mean', 'units'}
missing = required - set(zone_summary.columns)
if missing:
    raise ValueError(f'Missing columns in zone table: {sorted(missing)}')
zone_summary['year'] = zone_summary['year'].astype(int)
zone_summary['zone_mean'] = pd.to_numeric(zone_summary['zone_mean'], errors='coerce')

count_indices = ['R10mm', 'R20mm', 'CDD', 'CWD', 'SU', 'TR', 'FD', 'ID']
bad_duration = zone_summary[(zone_summary['index'].isin(count_indices)) & (zone_summary['zone_mean'].abs() > 1e5)]
if not bad_duration.empty:
    msg = f'Found {len(bad_duration)} implausibly large count/duration rows. Rerun notebook 07 count-index rebuild before using trend results.'
    print('WARNING:', msg)
    failures.append('DURATION_SCALE_WARNING|' + msg)

with xr.open_dataset(ZONES_FILE) as zds:
    zname = 'zone' if 'zone' in zds.data_vars else first_data_var(zds)
    zone_da = force_numeric_da(zds[zname]).load()

inv = []
for f in sorted(CORR_DIR.rglob('*.nc')):
    parts = f.relative_to(CORR_DIR).parts
    if len(parts) >= 3:
        m = re.search(r'_([^_]+)_annual_corrected\.nc$', f.name)
        inv.append({'scenario': parts[0], 'model': parts[1], 'index': m.group(1) if m else None, 'path': str(f.relative_to(ROOT)), 'size_mb': round(f.stat().st_size/1024/1024, 3)})
inventory = pd.DataFrame(inv)
inventory.to_csv(TABLE_DIR / 'etccdi_spatial_trend_input_inventory.csv', index=False)
print('Zone rows:', len(zone_summary), '| corrected files:', len(inventory), '| zone grid:', dict(zone_da.sizes))


## Cell 4 - Spatial Ensemble Change Maps


In [ ]:
spatial_records, spatial_failures = [], []
for idx in KEY_INDICES:
    for scenario in FUTURE_SCENARIOS:
        for period_name, (start, end) in PERIODS.items():
            model_changes, model_pct_changes, used_models = [], [], []
            for model in MODELS:
                if model in PRECIP_ONLY_MODELS and idx in TEMP_INDICES:
                    continue
                hist = load_corrected_index(model, 'historical', idx)
                fut = load_corrected_index(model, scenario, idx)
                hist_mean = period_mean(hist, *BASELINE)
                fut_mean = period_mean(fut, start, end)
                if hist_mean is None or fut_mean is None:
                    spatial_failures.append(f'MISSING|{model}|{scenario}|{period_name}|{idx}')
                    continue
                change = fut_mean - hist_mean
                model_changes.append(to_zone_grid(change, zone_da))
                model_pct_changes.append(to_zone_grid(safe_percent_change(change, hist_mean), zone_da))
                used_models.append(model)
            if not model_changes:
                spatial_failures.append(f'NO_MODELS|{scenario}|{period_name}|{idx}')
                continue
            stack = xr.concat(model_changes, dim=pd.Index(used_models, name='model'))
            pct_stack = xr.concat(model_pct_changes, dim=pd.Index(used_models, name='model'))
            ens = stack.mean('model', skipna=True).rename('ensemble_mean_change')
            std = stack.std('model', skipna=True).rename('model_std_change')
            pct = pct_stack.mean('model', skipna=True).rename('ensemble_mean_percent_change')
            nmod = stack.notnull().sum('model').rename('n_models')
            agreement = ((np.sign(stack) == np.sign(ens)).sum('model') / nmod * 100).rename('model_agreement_percent')
            out_ds = xr.Dataset({'ensemble_mean_change': ens.astype('float32'), 'ensemble_mean_percent_change': pct.astype('float32'), 'model_std_change': std.astype('float32'), 'model_agreement_percent': agreement.astype('float32'), 'n_models': nmod.astype('int16')})
            out_ds.attrs.update({'index': idx, 'scenario': scenario, 'period': period_name, 'period_years': f'{start}-{end}', 'baseline_years': f'{BASELINE[0]}-{BASELINE[1]}', 'units': INDEX_UNITS.get(idx, ''), 'models': ', '.join(used_models), 'method': 'future mean minus historical baseline; models regridded to hydroclimatic-zone grid before ensemble statistics'})
            out_nc = MAP_DIR / f'etccdi_{idx}_{scenario}_{period_name}_ensemble_change.nc'
            out_ds.to_netcdf(out_nc, encoding={v: {'zlib': True, 'complevel': 4} for v in out_ds.data_vars})
            spatial_records.append({'index': idx, 'scenario': scenario, 'period': period_name, 'start_year': start, 'end_year': end, 'n_models_used': len(used_models), 'models_used': ';'.join(used_models), 'mean_absolute_change_domain': float(ens.mean(skipna=True).values), 'mean_percent_change_domain': float(pct.mean(skipna=True).values), 'mean_model_agreement_percent': float(agreement.mean(skipna=True).values), 'map_path': str(out_nc.relative_to(ROOT)), 'units': INDEX_UNITS.get(idx, '')})
            print('[OK]', out_nc.relative_to(ROOT))
spatial_summary = pd.DataFrame(spatial_records)
spatial_summary.to_csv(TABLE_DIR / 'etccdi_spatial_change_map_inventory.csv', index=False)
(LOG_DIR / 'spatial_change_failures.txt').write_text('\n'.join(spatial_failures), encoding='utf-8')
print('Spatial products:', len(spatial_summary), '| failures/skips:', len(spatial_failures))
spatial_summary.head()


## Cell 5 - Publication Spatial Figures


In [ ]:
mpl.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 10, 'axes.titlesize': 11, 'axes.titleweight': 'bold', 'axes.labelsize': 10, 'axes.labelweight': 'bold', 'xtick.labelsize': 9, 'ytick.labelsize': 9, 'axes.spines.top': False, 'axes.spines.right': False, 'figure.dpi': 130})

def plot_map_panel(idx, scenario, period_name):
    path = MAP_DIR / f'etccdi_{idx}_{scenario}_{period_name}_ensemble_change.nc'
    if not path.exists():
        return None
    with xr.open_dataset(path) as ds:
        change = ds['ensemble_mean_change'].load()
        agree = ds['model_agreement_percent'].load()
    vmax = float(np.nanpercentile(np.abs(change.values), 98))
    vmax = 1.0 if (not np.isfinite(vmax) or vmax == 0) else vmax
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)
    im0 = axes[0].pcolormesh(change.lon, change.lat, change, cmap='RdBu_r', vmin=-vmax, vmax=vmax, shading='auto')
    axes[0].set_title('Ensemble mean change', loc='left')
    axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')
    cb0 = fig.colorbar(im0, ax=axes[0], shrink=0.88); cb0.set_label(f'{idx} change ({INDEX_UNITS.get(idx, "")})', fontweight='bold')
    im1 = axes[1].pcolormesh(agree.lon, agree.lat, agree, cmap='YlGnBu', vmin=0, vmax=100, shading='auto')
    axes[1].set_title('Model agreement on sign', loc='left')
    axes[1].set_xlabel('Longitude'); axes[1].set_ylabel('Latitude')
    cb1 = fig.colorbar(im1, ax=axes[1], shrink=0.88); cb1.set_label('Agreement (%)', fontweight='bold')
    for ax in axes:
        for tick in ax.get_xticklabels() + ax.get_yticklabels():
            tick.set_fontweight('bold')
    fig.suptitle(f'{INDEX_TITLES.get(idx, idx)} | {SCENARIO_LABELS[scenario]} | {period_name.replace("_", " ")}', fontsize=13, fontweight='bold')
    out_png = FIG_DIR / f'map_{idx}_{scenario}_{period_name}.png'
    out_pdf = FIG_DIR / f'map_{idx}_{scenario}_{period_name}.pdf'
    fig.savefig(out_png, dpi=300, bbox_inches='tight'); fig.savefig(out_pdf, bbox_inches='tight')
    plt.show()
    return out_png

made = []
for idx in KEY_INDICES:
    for scenario in FUTURE_SCENARIOS:
        for period_name in PERIODS:
            out = plot_map_panel(idx, scenario, period_name)
            if out is not None:
                made.append(str(out.relative_to(ROOT)))
pd.Series(made, name='figure_path').to_csv(TABLE_DIR / 'etccdi_spatial_figure_inventory.csv', index=False)
print('Spatial figures:', len(made))


## Cell 6 - Mann-Kendall, Sen Slope, and FDR Helpers


In [ ]:
def mann_kendall_test(y):
    y = np.asarray(y, dtype=float)
    y = y[np.isfinite(y)]
    n = len(y)
    if n < 8:
        return {'n': n, 's': np.nan, 'z': np.nan, 'p': np.nan, 'tau': np.nan, 'trend': 'insufficient'}
    s = sum(np.sign(y[k+1:] - y[k]).sum() for k in range(n - 1))
    _, counts = np.unique(y, return_counts=True)
    var_s = (n * (n - 1) * (2 * n + 5) - np.sum(counts * (counts - 1) * (2 * counts + 5))) / 18.0
    z = 0.0 if var_s <= 0 else ((s - 1) / math.sqrt(var_s) if s > 0 else ((s + 1) / math.sqrt(var_s) if s < 0 else 0.0))
    p = math.erfc(abs(z) / math.sqrt(2.0))
    tau = s / (0.5 * n * (n - 1))
    trend = 'increasing' if p < 0.05 and z > 0 else ('decreasing' if p < 0.05 and z < 0 else 'not_significant')
    return {'n': n, 's': float(s), 'z': float(z), 'p': float(p), 'tau': float(tau), 'trend': trend}

def sen_slope(x, y):
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]; y = y[mask]
    if len(y) < 2:
        return np.nan
    slopes = []
    for i in range(len(y) - 1):
        dx = x[i+1:] - x[i]
        valid = dx != 0
        slopes.extend(((y[i+1:] - y[i]) / dx)[valid])
    return float(np.median(slopes)) if slopes else np.nan

def benjamini_hochberg(pvals):
    pvals = np.asarray(pvals, dtype=float)
    out = np.full_like(pvals, np.nan, dtype=float)
    valid = np.isfinite(pvals)
    pv = pvals[valid]
    if len(pv) == 0:
        return out
    order = np.argsort(pv)
    ranked = pv[order]
    adj = ranked * len(pv) / np.arange(1, len(pv) + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    temp = np.empty_like(adj); temp[order] = np.clip(adj, 0, 1)
    out[valid] = temp
    return out


## Cell 7 - Zone-wise Trend Analysis


In [ ]:
trend_rows, trend_failures = [], []

def append_trend(series_type, model, scenario, idx, zone, zone_id, units, sub):
    sub = sub.sort_values('year')
    if len(sub) < 8:
        trend_failures.append(f'INSUFFICIENT|{series_type}|{model}|{scenario}|{idx}|{zone}|n={len(sub)}')
        return
    mk = mann_kendall_test(sub['zone_mean'].values)
    slope = sen_slope(sub['year'].values, sub['zone_mean'].values)
    trend_rows.append({'series_type': series_type, 'model': model, 'scenario': scenario, 'index': idx, 'zone': zone, 'zone_id': zone_id, 'start_year': int(sub['year'].min()), 'end_year': int(sub['year'].max()), 'n_years': mk['n'], 'sen_slope_per_year': slope, 'sen_slope_per_decade': slope * 10 if np.isfinite(slope) else np.nan, 'mk_tau': mk['tau'], 'mk_z': mk['z'], 'mk_p': mk['p'], 'mk_trend': mk['trend'], 'units': units})

for keys, sub in zone_summary.groupby(['model', 'scenario', 'index', 'zone', 'zone_id', 'units']):
    append_trend('model', *keys, sub)

ens = zone_summary.groupby(['scenario', 'index', 'zone', 'zone_id', 'units', 'year'], as_index=False)['zone_mean'].mean()
for keys, sub in ens.groupby(['scenario', 'index', 'zone', 'zone_id', 'units']):
    scenario, idx, zone, zone_id, units = keys
    append_trend('ensemble_mean', 'ensemble_mean', scenario, idx, zone, zone_id, units, sub)

trend_df = pd.DataFrame(trend_rows)
if not trend_df.empty:
    trend_df['mk_p_fdr'] = np.nan
    for _, locs in trend_df.groupby(['series_type', 'scenario', 'index']).groups.items():
        trend_df.loc[locs, 'mk_p_fdr'] = benjamini_hochberg(trend_df.loc[locs, 'mk_p'].values)
    trend_df['significant_fdr_0_05'] = trend_df['mk_p_fdr'] < 0.05
trend_df.to_csv(TABLE_DIR / 'etccdi_zone_mann_kendall_sen_trends.csv', index=False)
(LOG_DIR / 'trend_analysis_failures.txt').write_text('\n'.join(trend_failures), encoding='utf-8')
print('Trend rows:', len(trend_df), '| failures/skips:', len(trend_failures))
trend_df.head()


## Cell 8 - Zone Trend Heatmaps


In [ ]:
def plot_trend_heatmap(idx, scenario, series_type='ensemble_mean'):
    sub = trend_df[(trend_df['index'] == idx) & (trend_df['scenario'] == scenario) & (trend_df['series_type'] == series_type)]
    if sub.empty:
        return None
    values = sub.set_index('zone')['sen_slope_per_decade'].reindex([f'Z{i+1}' for i in ZONE_IDS])
    sig = sub.set_index('zone')['significant_fdr_0_05'].to_dict()
    vmax = np.nanpercentile(np.abs(values.values), 98)
    vmax = 1.0 if (not np.isfinite(vmax) or vmax == 0) else vmax
    fig, ax = plt.subplots(figsize=(5.2, 4.2))
    im = ax.imshow(values.values[:, None], cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')
    ax.set_xticks([0]); ax.set_xticklabels([SCENARIO_LABELS.get(scenario, scenario)], fontweight='bold')
    ax.set_yticks(range(len(values))); ax.set_yticklabels(values.index, fontweight='bold')
    ax.set_title(f'{INDEX_TITLES.get(idx, idx)}\nSen slope by zone', fontweight='bold')
    for y, zone in enumerate(values.index):
        val = values.loc[zone]
        label = '' if pd.isna(val) else f'{val:.2f}' + ('*' if sig.get(zone, False) else '')
        ax.text(0, y, label, ha='center', va='center', fontweight='bold', color='black')
    cb = fig.colorbar(im, ax=ax, shrink=0.85); cb.set_label(f'Change per decade ({INDEX_UNITS.get(idx, "")})', fontweight='bold')
    fig.tight_layout()
    out_png = FIG_DIR / f'trend_heatmap_{idx}_{scenario}_{series_type}.png'
    out_pdf = FIG_DIR / f'trend_heatmap_{idx}_{scenario}_{series_type}.pdf'
    fig.savefig(out_png, dpi=300, bbox_inches='tight'); fig.savefig(out_pdf, bbox_inches='tight')
    plt.show()
    return out_png

trend_figs = []
for idx in KEY_INDICES:
    for scenario in SCENARIOS:
        out = plot_trend_heatmap(idx, scenario)
        if out is not None:
            trend_figs.append(str(out.relative_to(ROOT)))
pd.Series(trend_figs, name='figure_path').to_csv(TABLE_DIR / 'etccdi_trend_figure_inventory.csv', index=False)
print('Trend figures:', len(trend_figs))


## Cell 9 - Results Tables


In [ ]:
ensemble_trends = trend_df[trend_df['series_type'] == 'ensemble_mean'].copy()
ensemble_trends[ensemble_trends['index'].isin(KEY_INDICES)].to_csv(TABLE_DIR / 'etccdi_key_indices_ensemble_zone_trends.csv', index=False)

sig_counts = trend_df.groupby(['series_type', 'scenario', 'index'], as_index=False).agg(n_tests=('mk_p', 'count'), n_fdr_significant=('significant_fdr_0_05', 'sum'), median_sen_slope_per_decade=('sen_slope_per_decade', 'median'))
sig_counts.to_csv(TABLE_DIR / 'etccdi_trend_significance_summary.csv', index=False)

spatial_summary[spatial_summary['index'].isin(KEY_INDICES)].to_csv(TABLE_DIR / 'etccdi_key_indices_spatial_change_summary.csv', index=False)

for f in ['etccdi_key_indices_ensemble_zone_trends.csv', 'etccdi_trend_significance_summary.csv', 'etccdi_key_indices_spatial_change_summary.csv']:
    print('[OK]', TABLE_DIR / f)


## Cell 10 - Save Phase Summary


In [ ]:
summary = f"""# Phase 4 ETCCDI Spatial and Trend Analysis Summary

## Purpose

This phase uses corrected annual ETCCDI products from notebook 07 to evaluate spatial climate-extreme changes and zone-wise temporal trends across hydroclimatic zones.

## Inputs

- Corrected annual ETCCDI NetCDFs: `{CORR_DIR.relative_to(ROOT)}`
- Zone annual means: `{ZONE_TABLE.relative_to(ROOT)}`
- Hydroclimatic zone raster: `{ZONES_FILE.relative_to(ROOT)}`

## Methods

- Spatial changes were calculated as future-period mean minus historical baseline mean.
- Baseline period: {BASELINE[0]}-{BASELINE[1]}.
- Future periods: {', '.join([f'{k} ({v[0]}-{v[1]})' for k, v in PERIODS.items()])}.
- Individual model change maps were interpolated to the hydroclimatic-zone grid before ensemble statistics.
- Spatial outputs include ensemble mean change, percent change, model standard deviation, number of contributing models, and model agreement on change sign.
- Zone-wise temporal trends were estimated using Mann-Kendall trend tests and Sen slope.
- Benjamini-Hochberg FDR correction was applied within each scenario-index trend-test family.

## Main Outputs

- Spatial map NetCDFs: `{MAP_DIR.relative_to(ROOT)}`
- Tables: `{TABLE_DIR.relative_to(ROOT)}`
- Figures: `{FIG_DIR.relative_to(ROOT)}`
- Logs: `{LOG_DIR.relative_to(ROOT)}`

## Key Tables

- `etccdi_spatial_change_map_inventory.csv`
- `etccdi_zone_mann_kendall_sen_trends.csv`
- `etccdi_key_indices_ensemble_zone_trends.csv`
- `etccdi_trend_significance_summary.csv`
- `etccdi_key_indices_spatial_change_summary.csv`

## Reviewer-Readiness Notes

This phase reports ensemble mean, model spread, model agreement, non-parametric trend significance, and FDR-adjusted significance. These diagnostics help separate robust projected signals from model-dependent or statistically weak changes.
"""
out = OUT_ROOT / 'PHASE_4_ETCCDI_SPATIAL_TREND_SUMMARY.md'
out.write_text(summary, encoding='utf-8')
print(out)
print(summary)


## Cell 11 - Manuscript and Supplementary Composite Figures

This cell does not change earlier outputs. It only creates compact figure plates suitable for the main manuscript and supplementary material.

Recommended manuscript use:
- Main Figure: far-future SSP5-8.5 spatial change for selected core indices.
- Supplementary Figure: near/far future spatial changes under both SSP2-4.5 and SSP5-8.5.


In [ ]:
# Composite figure settings.
MAIN_SPATIAL_INDICES = ['PRCPTOT', 'RX1day', 'CDD', 'TXx']
MAIN_SCENARIO = 'ssp585'
MAIN_PERIOD = 'far_future_2061_2100'
SUPP_SPATIAL_INDICES = ['PRCPTOT', 'RX1day', 'CDD', 'TXx', 'TNn']
SUPP_SCENARIOS = ['ssp245', 'ssp585']
SUPP_PERIODS = ['near_future_2021_2060', 'far_future_2061_2100']

mpl.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 10,
    'axes.titlesize': 10.5,
    'axes.titleweight': 'bold',
    'axes.labelsize': 9.5,
    'axes.labelweight': 'bold',
    'xtick.labelsize': 8.5,
    'ytick.labelsize': 8.5,
    'figure.dpi': 130,
})


def load_map_var(idx, scenario, period, var='ensemble_mean_change'):
    path = MAP_DIR / f'etccdi_{idx}_{scenario}_{period}_ensemble_change.nc'
    if not path.exists():
        return None
    with xr.open_dataset(path) as ds:
        return ds[var].load()


def robust_vlim(arrays, pct=98):
    vals = []
    for da in arrays:
        if da is not None:
            vals.append(np.ravel(da.values))
    if not vals:
        return 1.0
    vals = np.concatenate(vals)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return 1.0
    vmax = float(np.nanpercentile(np.abs(vals), pct))
    return 1.0 if (not np.isfinite(vmax) or vmax == 0) else vmax


def draw_change_map(ax, da, idx, title, vlim):
    if da is None:
        ax.axis('off')
        ax.set_title(title + '\nmissing', loc='left')
        return None
    im = ax.pcolormesh(da.lon, da.lat, da, cmap='RdBu_r', vmin=-vlim, vmax=vlim, shading='auto')
    ax.set_title(title, loc='left', pad=5)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.tick_params(length=2.5)
    for tick in ax.get_xticklabels() + ax.get_yticklabels():
        tick.set_fontweight('bold')
    return im

# Main manuscript spatial plate: selected indices, SSP5-8.5 far future.
main_arrays = [load_map_var(idx, MAIN_SCENARIO, MAIN_PERIOD) for idx in MAIN_SPATIAL_INDICES]
fig, axes = plt.subplots(2, 2, figsize=(12, 8.8), constrained_layout=True)
axes = axes.ravel()
panel_labels = list('ABCD')
for ax, idx, da, label in zip(axes, MAIN_SPATIAL_INDICES, main_arrays, panel_labels):
    vlim = robust_vlim([da])
    title = f'{label}. {INDEX_TITLES.get(idx, idx)}'
    im = draw_change_map(ax, da, idx, title, vlim)
    if im is not None:
        cb = fig.colorbar(im, ax=ax, shrink=0.82)
        cb.set_label(f'Change ({INDEX_UNITS.get(idx, "")})', fontweight='bold')
fig.suptitle('Projected changes in key climate-extreme indices, SSP5-8.5 far future (2061-2100)',
             fontsize=14, fontweight='bold')
main_png = FIG_DIR / 'MANUSCRIPT_Figure_ETCCDI_spatial_change_SSP585_far_future.png'
main_pdf = FIG_DIR / 'MANUSCRIPT_Figure_ETCCDI_spatial_change_SSP585_far_future.pdf'
fig.savefig(main_png, dpi=300, bbox_inches='tight')
fig.savefig(main_pdf, bbox_inches='tight')
plt.show()
print('[OK]', main_png.relative_to(ROOT))
print('[OK]', main_pdf.relative_to(ROOT))

# Supplementary spatial plate: all selected indices across both scenarios and periods.
for period in SUPP_PERIODS:
    fig, axes = plt.subplots(len(SUPP_SPATIAL_INDICES), len(SUPP_SCENARIOS), figsize=(12, 3.1 * len(SUPP_SPATIAL_INDICES)), constrained_layout=True)
    if len(SUPP_SPATIAL_INDICES) == 1:
        axes = np.array([axes])
    for r, idx in enumerate(SUPP_SPATIAL_INDICES):
        row_arrays = [load_map_var(idx, scenario, period) for scenario in SUPP_SCENARIOS]
        vlim = robust_vlim(row_arrays)
        for c, scenario in enumerate(SUPP_SCENARIOS):
            da = row_arrays[c]
            label = chr(65 + r * len(SUPP_SCENARIOS) + c)
            title = f'{label}. {INDEX_TITLES.get(idx, idx)} | {SCENARIO_LABELS.get(scenario, scenario)}'
            im = draw_change_map(axes[r, c], da, idx, title, vlim)
            if im is not None:
                cb = fig.colorbar(im, ax=axes[r, c], shrink=0.78)
                cb.set_label(f'{INDEX_UNITS.get(idx, "")}', fontweight='bold')
    fig.suptitle(f'Supplementary ETCCDI spatial changes: {period.replace("_", " ")} ', fontsize=14, fontweight='bold')
    supp_png = FIG_DIR / f'SUPPLEMENT_Figure_ETCCDI_spatial_change_{period}.png'
    supp_pdf = FIG_DIR / f'SUPPLEMENT_Figure_ETCCDI_spatial_change_{period}.pdf'
    fig.savefig(supp_png, dpi=300, bbox_inches='tight')
    fig.savefig(supp_pdf, bbox_inches='tight')
    plt.show()
    print('[OK]', supp_png.relative_to(ROOT))
    print('[OK]', supp_pdf.relative_to(ROOT))


## Cell 12 - Main and Supplementary Trend Plates

This cell compacts the trend heatmaps into one manuscript-ready plate and one supplementary plate.
- Main plate: key indices under SSP5-8.5.
- Supplementary plate: key indices under Historical, SSP2-4.5, and SSP5-8.5.

Asterisks mark FDR-adjusted significance at 0.05.


In [ ]:
MAIN_TREND_INDICES = ['PRCPTOT', 'RX1day', 'CDD', 'TXx', 'TNn']
MAIN_TREND_SCENARIO = 'ssp585'
SUPP_TREND_SCENARIOS = ['historical', 'ssp245', 'ssp585']


def trend_values(idx, scenario):
    sub = trend_df[(trend_df['series_type'] == 'ensemble_mean') & (trend_df['index'] == idx) & (trend_df['scenario'] == scenario)].copy()
    values = sub.set_index('zone')['sen_slope_per_decade'].reindex([f'Z{i+1}' for i in ZONE_IDS])
    sig = sub.set_index('zone')['significant_fdr_0_05'].reindex([f'Z{i+1}' for i in ZONE_IDS]).fillna(False)
    return values, sig


def draw_trend_cell(ax, values, sig, idx, title, vlim):
    im = ax.imshow(values.values[:, None], cmap='RdBu_r', vmin=-vlim, vmax=vlim, aspect='auto')
    ax.set_xticks([0])
    ax.set_xticklabels(['Slope'], fontweight='bold')
    ax.set_yticks(range(len(values)))
    ax.set_yticklabels(values.index, fontweight='bold')
    ax.set_title(title, loc='left', pad=5)
    for y, zone in enumerate(values.index):
        val = values.loc[zone]
        txt = '' if pd.isna(val) else f'{val:.2f}' + ('*' if bool(sig.loc[zone]) else '')
        ax.text(0, y, txt, ha='center', va='center', fontsize=8.5, fontweight='bold')
    return im

# Main manuscript trend plate.
all_vals = [trend_values(idx, MAIN_TREND_SCENARIO)[0] for idx in MAIN_TREND_INDICES]
vlim = robust_vlim([xr.DataArray(v.values) for v in all_vals])
fig, axes = plt.subplots(1, len(MAIN_TREND_INDICES), figsize=(15, 4.5), constrained_layout=True)
for ax, idx, label in zip(axes, MAIN_TREND_INDICES, list('ABCDE')):
    values, sig = trend_values(idx, MAIN_TREND_SCENARIO)
    im = draw_trend_cell(ax, values, sig, idx, f'{label}. {idx}', vlim)
cb = fig.colorbar(im, ax=axes, shrink=0.82, location='right')
cb.set_label('Sen slope per decade', fontweight='bold')
fig.suptitle('Zone-wise trends in key ETCCDI indices, SSP5-8.5', fontsize=14, fontweight='bold')
out_png = FIG_DIR / 'MANUSCRIPT_Figure_ETCCDI_zone_trends_SSP585.png'
out_pdf = FIG_DIR / 'MANUSCRIPT_Figure_ETCCDI_zone_trends_SSP585.pdf'
fig.savefig(out_png, dpi=300, bbox_inches='tight')
fig.savefig(out_pdf, bbox_inches='tight')
plt.show()
print('[OK]', out_png.relative_to(ROOT))
print('[OK]', out_pdf.relative_to(ROOT))

# Supplementary trend plate.
fig, axes = plt.subplots(len(MAIN_TREND_INDICES), len(SUPP_TREND_SCENARIOS), figsize=(12, 2.65 * len(MAIN_TREND_INDICES)), constrained_layout=True)
for r, idx in enumerate(MAIN_TREND_INDICES):
    row_vals = [trend_values(idx, sc)[0] for sc in SUPP_TREND_SCENARIOS]
    row_vlim = robust_vlim([xr.DataArray(v.values) for v in row_vals])
    for c, scenario in enumerate(SUPP_TREND_SCENARIOS):
        values, sig = trend_values(idx, scenario)
        label = chr(65 + r * len(SUPP_TREND_SCENARIOS) + c)
        im = draw_trend_cell(axes[r, c], values, sig, idx, f'{label}. {idx} | {SCENARIO_LABELS.get(scenario, scenario)}', row_vlim)
        if c > 0:
            axes[r, c].set_yticklabels([])
fig.suptitle('Supplementary zone-wise ETCCDI trend diagnostics', fontsize=14, fontweight='bold')
out_png = FIG_DIR / 'SUPPLEMENT_Figure_ETCCDI_zone_trends_all_scenarios.png'
out_pdf = FIG_DIR / 'SUPPLEMENT_Figure_ETCCDI_zone_trends_all_scenarios.pdf'
fig.savefig(out_png, dpi=300, bbox_inches='tight')
fig.savefig(out_pdf, bbox_inches='tight')
plt.show()
print('[OK]', out_png.relative_to(ROOT))
print('[OK]', out_pdf.relative_to(ROOT))


## Cell 13 - Export Spatial Raster Results as GeoTIFF for QGIS

This cell exports every spatial NetCDF result from Cell 4 to GeoTIFF. Each NetCDF variable is exported separately:
- `ensemble_mean_change`
- `ensemble_mean_percent_change`
- `model_std_change`
- `model_agreement_percent`
- `n_models`

The exported rasters are placed under `output/etccdi_spatial_trends/geotiff/` and can be opened directly in QGIS.


In [ ]:
GEOTIFF_DIR = OUT_ROOT / 'geotiff'
GEOTIFF_DIR.mkdir(parents=True, exist_ok=True)

try:
    import rasterio
    from rasterio.transform import from_bounds
except Exception as exc:
    raise ImportError('GeoTIFF export needs rasterio in the geo environment. Install rasterio, then rerun this cell.') from exc


def export_da_to_geotiff(da, out_tif, nodata=-9999.0):
    da = standardise_xy(da.squeeze(drop=True))
    if 'lat' not in da.coords or 'lon' not in da.coords:
        raise ValueError('DataArray must have lat/lon coordinates for GeoTIFF export.')
    # GeoTIFF convention expects north-up row order.
    if float(da.lat[0]) < float(da.lat[-1]):
        da = da.sortby('lat', ascending=False)
    arr = da.values.astype('float32')
    arr = np.where(np.isfinite(arr), arr, nodata).astype('float32')
    lon = da.lon.values
    lat = da.lat.values
    xres = abs(float(lon[1] - lon[0])) if len(lon) > 1 else 0.25
    yres = abs(float(lat[1] - lat[0])) if len(lat) > 1 else 0.25
    west = float(lon.min()) - xres / 2
    east = float(lon.max()) + xres / 2
    south = float(lat.min()) - yres / 2
    north = float(lat.max()) + yres / 2
    transform = from_bounds(west, south, east, north, arr.shape[1], arr.shape[0])
    out_tif.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(
        out_tif, 'w', driver='GTiff', height=arr.shape[0], width=arr.shape[1], count=1,
        dtype='float32', crs='EPSG:4326', transform=transform, nodata=nodata,
        compress='deflate', predictor=2
    ) as dst:
        dst.write(arr, 1)
        dst.set_band_description(1, da.name or out_tif.stem)

exports = []
for nc_path in sorted(MAP_DIR.glob('etccdi_*_ensemble_change.nc')):
    with xr.open_dataset(nc_path) as ds:
        idx = ds.attrs.get('index', 'unknown')
        scenario = ds.attrs.get('scenario', 'unknown')
        period = ds.attrs.get('period', 'unknown')
        for var in ds.data_vars:
            da = ds[var].load().rename(var)
            out_tif = GEOTIFF_DIR / scenario / period / idx / f'{idx}_{scenario}_{period}_{var}.tif'
            export_da_to_geotiff(da, out_tif)
            exports.append({'index': idx, 'scenario': scenario, 'period': period, 'variable': var, 'path': str(out_tif.relative_to(ROOT))})
            print('[OK]', out_tif.relative_to(ROOT))

geotiff_inventory = pd.DataFrame(exports)
geotiff_inventory.to_csv(TABLE_DIR / 'etccdi_spatial_geotiff_inventory.csv', index=False)
print('GeoTIFFs exported:', len(geotiff_inventory))


## Cell 14 - Update Figure and GeoTIFF Summary


In [ ]:
extra_summary = f"""# Additional Manuscript/Supplementary Figure and GeoTIFF Outputs

## Manuscript Composite Figures

- `figures/MANUSCRIPT_Figure_ETCCDI_spatial_change_SSP585_far_future.png`
- `figures/MANUSCRIPT_Figure_ETCCDI_spatial_change_SSP585_far_future.pdf`
- `figures/MANUSCRIPT_Figure_ETCCDI_zone_trends_SSP585.png`
- `figures/MANUSCRIPT_Figure_ETCCDI_zone_trends_SSP585.pdf`

## Supplementary Composite Figures

- `figures/SUPPLEMENT_Figure_ETCCDI_spatial_change_near_future_2021_2060.png`
- `figures/SUPPLEMENT_Figure_ETCCDI_spatial_change_near_future_2021_2060.pdf`
- `figures/SUPPLEMENT_Figure_ETCCDI_spatial_change_far_future_2061_2100.png`
- `figures/SUPPLEMENT_Figure_ETCCDI_spatial_change_far_future_2061_2100.pdf`
- `figures/SUPPLEMENT_Figure_ETCCDI_zone_trends_all_scenarios.png`
- `figures/SUPPLEMENT_Figure_ETCCDI_zone_trends_all_scenarios.pdf`

## QGIS Raster Outputs

GeoTIFF exports are saved under:

- `geotiff/`

Each spatial NetCDF variable is exported separately for direct QGIS mapping and cartographic refinement.
"""
summary_path = OUT_ROOT / 'PHASE_4_ADDITIONAL_FIGURES_AND_GEOTIFFS.md'
summary_path.write_text(extra_summary, encoding='utf-8')
print(summary_path)
print(extra_summary)
